In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, LSTM, GRU, Bidirectional, Conv1D,
                                     GlobalAveragePooling1D, Reshape, Input,
                                     Dropout, MultiHeadAttention, LayerNormalization, Flatten)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, matthews_corrcoef,
                             log_loss, cohen_kappa_score, brier_score_loss)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
df = pd.read_csv('file1.csv')

# Define target and features based on Dataset 1 logic
target = 'Diagnostic'
X = df.drop(['UDI', 'Product ID', 'Date', target], axis=1, errors='ignore')
y = (df[target] != 'No failure').astype(int)

numeric_features = X.select_dtypes(include=['int64', 'float64', 'number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Sequential([Input(shape=(len(numeric_features),)), Dense(len(numeric_features))]), numeric_features), # Placeholder to show logic
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Re-defining preprocessor more robustly for the loop
def get_preprocessor():
    return ColumnTransformer([
        ('num', Sequential([Input(shape=(len(numeric_features),))]), numeric_features), # Dummy just to list cols
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ], remainder='passthrough')

# Since SimpleImputer and Scaler are needed:
num_pipe = tf.keras.utils.register_keras_serializable()(lambda: None) # Not needed for sklearn
from sklearn.pipeline import Pipeline
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scal', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

# ==========================================
# 2. DEEP LEARNING MODEL BUILDER
# ==========================================
def build_dl_model(name, input_dim):
    if name == 'Transformer':
        inputs = Input(shape=(input_dim,))
        x = Reshape((input_dim, 1))(inputs)
        # Basic Transformer Block
        attn_output = MultiHeadAttention(num_heads=2, key_dim=2)(x, x)
        attn_output = Dropout(0.1)(attn_output)
        out1 = LayerNormalization(epsilon=1e-6)(x + attn_output)
        x = Flatten()(out1)
        x = Dense(32, activation='relu')(x)
        outputs = Dense(1, activation='sigmoid')(x)
        model = Model(inputs=inputs, outputs=outputs)
    else:
        model = Sequential([Input(shape=(input_dim,))])
        if name == '1D-CNN':
            model.add(Reshape((input_dim, 1)))
            model.add(Conv1D(64, kernel_size=3, activation='relu'))
            model.add(GlobalAveragePooling1D())
        elif name in ['LSTM', 'GRU', 'BiLSTM']:
            model.add(Reshape((input_dim, 1)))
            if name == 'LSTM': model.add(LSTM(64))
            elif name == 'GRU': model.add(GRU(64))
            elif name == 'BiLSTM': model.add(Bidirectional(LSTM(64)))

        model.add(Dense(32, activation='relu'))
        model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

# ==========================================
# 3. EVALUATION LOOP (10 METRICS x 5-FOLD CV)
# ==========================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_names = ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM',
               '1D-CNN', 'LSTM', 'GRU', 'BiLSTM', 'Transformer']
final_results = []

for name in model_names:
    print(f"Processing Model: {name}...")
    m_store = {k: [] for k in ['acc', 'f1', 'prec', 'rec', 'spec', 'auc', 'mcc', 'kappa', 'brier', 'logloss']}

    for train_idx, val_idx in skf.split(X, y):
        Xt_raw, Xv_raw = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]

        # 1. Preprocess
        Xt_p = preprocessor.fit_transform(Xt_raw)
        Xv_p = preprocessor.transform(Xv_raw)

        # 2. Oversample (SMOTE)
        X_res, y_res = SMOTE(random_state=42).fit_resample(Xt_p, yt)

        # 3. Instantiate and Train
        if name == 'Logistic Regression':
            clf = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_res, y_res)
        elif name == 'Random Forest':
            clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_res, y_res)
        elif name == 'XGBoost':
            clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42).fit(X_res, y_res)
        elif name == 'LightGBM':
            clf = LGBMClassifier(random_state=42, verbose=-1).fit(X_res, y_res)
        else:
            # DL Models
            clf = build_dl_model(name, X_res.shape[1])
            clf.fit(X_res, y_res, epochs=10, batch_size=32, verbose=0)

        # 4. Predictions
        if name in ['1D-CNN', 'LSTM', 'GRU', 'BiLSTM', 'Transformer']:
            y_prob = clf.predict(Xv_p).flatten()
        else:
            y_prob = clf.predict_proba(Xv_p)[:, 1]

        y_pred = (y_prob > 0.5).astype(int)
        tn, fp, fn, tp = confusion_matrix(yv, y_pred).ravel()

        # 5. Log 10 Metrics
        m_store['acc'].append(accuracy_score(yv, y_pred))
        m_store['f1'].append(f1_score(yv, y_pred))
        m_store['prec'].append(precision_score(yv, y_pred))
        m_store['rec'].append(recall_score(yv, y_pred))
        m_store['spec'].append(tn / (tn + fp))
        m_store['auc'].append(roc_auc_score(yv, y_prob))
        m_store['mcc'].append(matthews_corrcoef(yv, y_pred))
        m_store['kappa'].append(cohen_kappa_score(yv, y_pred))
        m_store['brier'].append(brier_score_loss(yv, y_prob))
        m_store['logloss'].append(log_loss(yv, y_prob))

    # 6. Compile Result Row (Mean ± Std)
    row = {'Model': name}
    for k in m_store:
        row[k] = f"{np.mean(m_store[k]):.4f} ± {np.std(m_store[k]):.4f}"
    final_results.append(row)

# ==========================================
# 4. SUMMARY TABLE
# ==========================================
summary_df = pd.DataFrame(final_results)
summary_df.columns = ['Model', 'Accuracy', 'F1 Score', 'Precision', 'Recall',
                      'Specificity', 'AUC', 'MCC', 'Kappa', 'Brier Score', 'Log-Loss']

print("\n" + "="*120)
print("INTEGRATED PERFORMANCE SUMMARY (BASELINE DATASET 1)")
print("="*120)
print(summary_df.to_string(index=False))

Processing Model: Logistic Regression...
Processing Model: Random Forest...
Processing Model: XGBoost...
Processing Model: LightGBM...
Processing Model: 1D-CNN...
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Processing Model: LSTM...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Processing Model: GRU...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Processing Model: BiLSTM...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step
63/63 ━━━━━━